# 02. Avaluació del Model Naive (Línia Base)

En aquest quadern implemente el model Naive (*baseline* o model ingenu) com a punt de partida per a l'estudi. Configure l'entorn de treball i els horitzons temporals de predicció (15, 30, 45 i 60 minuts) per a establir una línia base de referència sòlida amb la qual es compararan els models complexos posteriors.

In [1]:
# Importació de llibreries i configuració inicial
import pandas as pd
import numpy as np
import os
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Configuració dels horitzons de predicció (en passos de 5 minuts)
horitzons = {
    '15_min': 3,
    '30_min': 6,
    '45_min': 9,
    '60_min': 12
}

hores_warm_up = 11
warm_up_mostres = int((hores_warm_up * 60) / 5)

print(f"Configuració inicialitzada: Warm-up de {warm_up_mostres} mostres. Horitzons: {list(horitzons.keys())}")

Configuració inicialitzada: Warm-up de 132 mostres. Horitzons: ['15_min', '30_min', '45_min', '60_min']


### 1. Càrrega de Dades i Validació del Llindar Crític

Carregue el conjunt de dades net de test. Incorpore una verificació de seguretat (*assert*) per a assegurar que el fitxer compleix el requisit temporal mínim de 12 hores (144 mostres), garantint que la consistència estructural de les dades siga idèntica a la dels models ARIMA i ARIMAX.

In [2]:
ruta_test = "dades_preprocessades/dataset_test_net.csv"

if not os.path.exists(ruta_test):
    print(f"Error: No s'ha trobat l'arxiu a {ruta_test}")
else:
    df_test = pd.read_csv(ruta_test)
    df_test['timestamp'] = pd.to_datetime(df_test['timestamp'])
    
    # Verificació de seguretat del llindar crític
    min_mostres_test = df_test.groupby(['pacient_id', 'segment_num']).size().min()
    assert min_mostres_test >= 144, f"Error: Hi ha segments curts ({min_mostres_test} mostres). El mínim requerit és 144."
    print("✅ Verificació correcta: Tots els segments compleixen el llindar de 12h.")

✅ Verificació correcta: Tots els segments compleixen el llindar de 12h.


### 2. Metodologia del Model Naive i Acumulació de Prediccions

Implemente la lògica del model de desplaçament (*lagged*). Per a cada pacient i segment, desplace els valors reals segons l'horitzó temporal corresponent, descartant els punts del *warm-up* inicial per replicar el comportament de l'ARIMA. Acumule totes les prediccions en un repositori global abans d'efectuar cap càlcul estadístic.

In [3]:
# Diccionari per a acumular TOTS els valors reals i prediccions per horitzó
dades_globals = {h: {'y_true': [], 'y_pred': []} for h in horitzons.keys()}

print("Iniciant avaluació del Model Naive...")

for (pacient, segment), df_seg in df_test.groupby(['pacient_id', 'segment_num']):
    df_seg = df_seg.sort_values('timestamp').reset_index(drop=True)
    y_real = df_seg['glucose_imputed'].values
    N = len(y_real)
    
    for nom_h, passos in horitzons.items():
        if N > warm_up_mostres + passos:
            # El valor present s'utilitza com a predicció de futur (Naive)
            y_pred = y_real[warm_up_mostres : N - passos]
            # El valor futur real amb el qual es compararà
            y_true = y_real[warm_up_mostres + passos : N]
            
            # Acumule els vectors en les llistes globals
            dades_globals[nom_h]['y_true'].extend(y_true)
            dades_globals[nom_h]['y_pred'].extend(y_pred)
            
print("Acumulació completada. Processant mètriques globals...")

Iniciant avaluació del Model Naive...
Acumulació completada. Processant mètriques globals...


### 3. Disseny de Mètriques amb Protecció Matemàtica

Definisc la funció d'avaluació global per al càlcul de l'RMSE, el MAE i el MARD. Incorpore una protecció matemàtica específica mitjançant vectors de substitució de zeros per valors nuls (`np.where`). Aquest disseny metodològic prevé l'error de divisió per zero en el càlcul del MARD clínic en cas d'existir valors homogenis extrems.

In [4]:
def calcular_metriques_segures(real, pred):
    rmse = np.sqrt(mean_squared_error(real, pred))
    mae = mean_absolute_error(real, pred)
    
    # Protecció davant d'una impossible divisió per zero en la mètrica MARD
    real_segur = np.where(real == 0, np.nan, real)
    mard = np.nanmean(np.abs((real - pred) / real_segur)) * 100
    return rmse, mae, mard

### 4. Processament de Resultats Globals i Exportació

S'executa el càlcul final agregant el conjunt complet de prediccions pool per a garantir una comparativa matemàticament simètrica amb els models complexos. Es dona format de visualització amb arredoniment a dos decimals i s'exporten els indicadors calculats cap a un fitxer CSV de resultats metrics.

In [5]:
if 'dades_globals' in locals():
    resultats_finals = []
    
    for nom_h in horitzons.keys():
        y_t = np.array(dades_globals[nom_h]['y_true'])
        y_p = np.array(dades_globals[nom_h]['y_pred'])
        
        if len(y_t) > 0:
            rmse, mae, mard = calcular_metriques_segures(y_t, y_p)
            resultats_finals.append({
                'Horitzó': nom_h,
                'RMSE': rmse,
                'MAE': mae,
                'MARD (%)': mard,
                'Num_Prediccions': len(y_t)
            })
            
    df_global_metrics = pd.DataFrame(resultats_finals)
    
    # Ordenació i indexació de categories
    ordre_horitzons = ['15_min', '30_min', '45_min', '60_min']
    df_global_metrics['Horitzó'] = pd.Categorical(df_global_metrics['Horitzó'], categories=ordre_horitzons, ordered=True)
    df_global_metrics = df_global_metrics.sort_values('Horitzó')
    
    df_global_metrics['RMSE'] = df_global_metrics['RMSE'].round(2)
    df_global_metrics['MAE'] = df_global_metrics['MAE'].round(2)
    df_global_metrics['MARD (%)'] = df_global_metrics['MARD (%)'].round(2)

    print("\n" + "="*70)
    print(" RESULTATS GLOBALS DEL MODEL NAIVE (BASELINE) - FORMAT UNIFICAT")
    print("="*70)
    print(df_global_metrics.to_string(index=False))

    # Exportació definitiva de la línia base
    carpeta_resultats = "resultats_metrics"
    os.makedirs(carpeta_resultats, exist_ok=True)
    ruta_exportacio = os.path.join(carpeta_resultats, "metriques_naive.csv")
    df_global_metrics.to_csv(ruta_exportacio, index=False)
    
    print(f"\nS'han exportat les mètriques base correctament a '{ruta_exportacio}'.")


 RESULTATS GLOBALS DEL MODEL NAIVE (BASELINE) - FORMAT UNIFICAT
Horitzó  RMSE   MAE  MARD (%)  Num_Prediccions
 15_min 13.49  9.29      6.04            11642
 30_min 22.29 16.06     10.44            11549
 45_min 29.49 21.66     14.14            11456
 60_min 35.52 26.32     17.31            11363

S'han exportat les mètriques base correctament a 'resultats_metrics\metriques_naive.csv'.


### 5. Avaluació Clínica: Detecció d'Hipoglucèmies

Per a quantificar la seguretat clínica de la línia base, traduïsc el problema de regressió contínua a un problema de classificació binària. Establisc el llindar internacional d'hipoglucèmia en $\le 70$ mg/dL i avalue les mètriques de classificació estàndard (Sensibilitat, Precisió i F1-Score). Aquesta avaluació es realitza a nivell de finestra de predicció, reflectint la cobertura temporal en lloc del recompte absolut d'episodis clínics. Finalment, els indicadors calculats s'exporten per a la comparativa global.

In [7]:
LLINDAR_HIPO = 70.0

# Mapeig entre els horitzons del Naive (claus de dades_globals) i les etiquetes
horitzons_map = {
    '15_min': '15m',
    '30_min': '30m',
    '45_min': '45m',
    '60_min': '60m'
}

resultats_hipo = []

for clau_globals, etiqueta in horitzons_map.items():
    # Extracció de matrius per a l'horitzó corresponent
    y_true = np.array(dades_globals[clau_globals]['y_true'])
    y_pred = np.array(dades_globals[clau_globals]['y_pred'])

    # Binarització de les prediccions segons el llindar clínic
    real_hipo = (y_true <= LLINDAR_HIPO)
    pred_hipo = (y_pred <= LLINDAR_HIPO)

    # Càlcul de la Matriu de Confusió
    tp = np.sum(real_hipo & pred_hipo)        # Veritables Positius
    fn = np.sum(real_hipo & ~pred_hipo)       # Falsos Negatius
    fp = np.sum(~real_hipo & pred_hipo)       # Falsos Positius
    tn = np.sum(~real_hipo & ~pred_hipo)      # Veritables Negatius

    # Càlcul de mètriques amb prevenció de divisió per zero
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1        = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    # Acumulació de resultats
    resultats_hipo.append({
        'Horitzó'                   : etiqueta,
        'Casos_Hipo_Reals'          : int(np.sum(real_hipo)),
        'TP (Detectats)'            : int(tp),
        'FN (Omesos)'               : int(fn),
        'FP (Falses Alarmes)'       : int(fp),
        'Sensibilitat / Recall (%)' : round(recall * 100, 2),
        'Precisió (%)'              : round(precision * 100, 2),
        'F1-Score (%)'              : round(f1 * 100, 2)
    })

df_hipo_naive = pd.DataFrame(resultats_hipo)

print("\n" + "="*80)
print(" TAULA 3: DETECCIÓ D'HIPOGLUCÈMIES — MODEL NAIVE")
print("="*80)
print(df_hipo_naive.to_string(index=False))

# Exportació dels resultats de classificació
df_hipo_naive.to_csv("resultats_metrics/naive_hipoglucemies.csv", index=False)
print("\nExportació de mètriques clíniques completada.")


 TAULA 3: DETECCIÓ D'HIPOGLUCÈMIES — MODEL NAIVE
Horitzó  Casos_Hipo_Reals  TP (Detectats)  FN (Omesos)  FP (Falses Alarmes)  Sensibilitat / Recall (%)  Precisió (%)  F1-Score (%)
    15m               249             172           77                   71                      69.08         70.78         69.92
    30m               249             122          127                  116                      49.00         51.26         50.10
    45m               248              86          162                  150                      34.68         36.44         35.54
    60m               245              59          186                  177                      24.08         25.00         24.53

Exportació de mètriques clíniques completada.
